# L4 · MMMU 900문항 · 출력 2,048토큰 실제 실행

기존 32,768토큰 실행과 비교하는 새 독립 실행입니다. 출력 한도만 2,048로 변경하며 전체 컨텍스트 40,960, L4 BF16, 모델·데이터 revision, P0 프롬프트, 이미지 처리, sampling과 문항별 seed, 동시 처리 2개, V8 채점은 유지합니다.

JOB_ID: `mmmu-l4-2048-01`. 기존 `mmmu-l4-full-01` 결과를 재사용하거나 덮어쓰지 않습니다. 저장 답변 절단 분석과 구분되는 새 GPU 추론입니다.

원본 엔진은 `f1235cf76f1c5637ea61b72a7d36b01c8fbc73a1`의 깨끗한 checkout을 유지합니다. Colab adapter가 `mmmu-val-v8-2048` 설정을 기존 설정과 비교해 출력 한도 외의 변경을 거부하며, 새로운 adapter hash도 실행 신원에 포함합니다.

마지막 셀은 입력 검사 → 900문항 추론 → 원본 검증 → 별도 V8 재채점·검증 → Drive 최종 백업 → 런타임 해제를 수행합니다. 실패해도 백업 성공 시 종료하며, 백업 실패 시 결과를 보존하기 위해 유지합니다. 모든 증분 ZIP과 ready marker를 함께 보존하세요. 실행 중인 셀을 다시 실행하지 마세요.


## 1. GPU 확인 — 설치·다운로드 전에 실행

In [ ]:
import subprocess, shutil, json, os, sys, re
from pathlib import Path

def call(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

query = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
    '--format=csv,noheader,nounits'], text=True).strip()
print(query)
name, memory, capability = [x.strip() for x in query.splitlines()[0].split(',')]
assert name == 'NVIDIA L4', '비교 실행은 NVIDIA L4만 허용합니다.'
assert float(capability) >= 8.0, 'T4 등 native BF16 미지원 GPU입니다. L4 또는 A100 런타임을 선택하세요.'
assert int(memory) >= 22000, '이 노트북은 현재 저장소의 24GB 이상 설정을 유지합니다.'
assert shutil.disk_usage('/content').free / 2**30 >= 85, '로컬 디스크 여유 85GiB 이상을 권장합니다. 저장소 자체도 실행 전 여유 공간을 검사합니다.'
GPU_NAME, GPU_MIB = name, int(memory)
print('GPU 조건 통과. 전체 실행 전 GPU와 백업 상태를 확인하세요.')

## 2. Drive 연결 및 작업 이름
Drive 접근 권한은 본인이 승인하세요. 다음 세션에도 **동일한 JOB_ID**를 사용합니다. 새 독립 실험일 때만 이름을 바꾸세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
JOB_ID = 'mmmu-l4-2048-01'
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,45}', JOB_ID)
REMOTE = Path('/content/drive/MyDrive/MMDL-Colab') / JOB_ID
REMOTE.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content') / ('mmdl-artifacts-' + JOB_ID)
REPO = Path('/content/MMDL')
COMMIT = 'f1235cf76f1c5637ea61b72a7d36b01c8fbc73a1'
if not REPO.exists():
    call(['git', 'clone', 'https://github.com/jang2296/MMDL.git', REPO])
call(['git', 'checkout', '--detach', COMMIT], cwd=REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip() == COMMIT
print('백업 폴더:', REMOTE)

## 3. 고정 Python 환경 설치
코랩 기본 패키지를 덮어쓰지 않고 별도 환경을 만듭니다. 새 VM에서는 다시 설치합니다. 설치 실패 시 버전을 임의로 풀지 말고 오류를 확인하세요.

In [ ]:
def call(args, **kwargs):
    args = [str(x) for x in args]
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, args)

call([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
VENV = Path('/content/mmdl-env')
if not VENV.exists():
    call([UV, 'venv', '--python', '3.12.3', VENV])
PYTHON = VENV / 'bin/python'
call([UV, 'pip', 'install', '--python', PYTHON, 'pip==25.2'])
call([PYTHON, '-m', 'pip', 'install', '-r', REPO/'env/requirements-vllm.lock'])
call([PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', REPO])
call([PYTHON, '-c', 'import torch; print(torch.__version__, torch.cuda.get_device_name()); assert torch.cuda.is_bf16_supported()'])
os.environ.update(HF_HOME='/content/mmdl-cache/huggingface',
    MMDL_DATA_ROOT='/content/mmdl-data', MMDL_ARTIFACT_ROOT=str(LOCAL),
    MMDL_COLAB_BACKUP=str(REMOTE), CUDA_VISIBLE_DEVICES='0',
    TOKENIZERS_PARALLELISM='false', VLLM_WORKER_MULTIPROC_METHOD='spawn')

## 4. 세션 백업 도구 설치 및 복원
아래 코드는 노트북에 포함되어 있어 별도 파일 업로드가 필요 없습니다. Drive에는 검증 완료 표시가 있는 ZIP만 복원합니다. 중단된 ZIP은 무시하고, 해시가 다르면 멈춥니다.

In [ ]:
RUNNER_SOURCE = '"""Colab adapter: append-only verified checkpoints; original evaluation stays pinned.\n\nOnly the host kernel release is excluded from resume identity. GPU, driver,\nPython, installed packages, protocol and original source identity remain strict.\nRun one notebook per backup directory. This is not a distributed writer.\n"""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport tempfile\nimport uuid\nimport zipfile\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef checkpoints(remote):\n    markers = sorted(Path(remote).glob(\'*.ready.json\'))\n    entries = []\n    for index, path in enumerate(markers, 1):\n        entry = json.loads(path.read_text())\n        if entry[\'sequence\'] != index or not path.name.startswith(f\'{index:06d}-\'):\n            raise RuntimeError(\'Checkpoint gap or concurrent writers; preserve Drive files and inspect.\')\n        if Path(entry[\'archive\']).name != entry[\'archive\']:\n            raise ValueError(\'Unsafe checkpoint archive path\')\n        entries.append(entry)\n    return entries\n\n\ndef restore(remote, local):\n    local = Path(local)\n    if local.exists():\n        raise FileExistsError(\'Restore requires a fresh local directory; existing work is preserved.\')\n    entries = checkpoints(remote)\n    local.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=local.parent) as td:\n        stage = Path(td) / \'artifacts\'\n        stage.mkdir()\n        for entry in entries:\n            archive = Path(td) / \'checkpoint.zip\'\n            shutil.copyfile(Path(remote) / entry[\'archive\'], archive)\n            if sha(archive) != entry[\'sha256\']:\n                raise ValueError(f"Checkpoint checksum failed: {entry[\'archive\']}")\n            with zipfile.ZipFile(archive) as z:\n                for info in z.infolist():\n                    relative = Path(info.filename)\n                    if relative.is_absolute() or \'..\' in relative.parts or info.is_dir():\n                        raise ValueError(\'Unsafe checkpoint entry\')\n                    target = stage / relative\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with z.open(info) as src, target.open(\'wb\') as dst:\n                        shutil.copyfileobj(src, dst)\n            archive.unlink()\n        stage.rename(local)\n    print(f\'Restored {len(entries)} verified checkpoints\', flush=True)\n\n\nclass CheckpointStore:\n    def __init__(self, local, remote):\n        self.local, self.remote = Path(local), Path(remote)\n        self.remote.mkdir(parents=True, exist_ok=True)\n        self.sequence = len(checkpoints(self.remote))\n        self.seen = {}\n\n    def save(self):\n        # Called synchronously after a complete sample, never from a background thread.\n        changed, stamps = [], {}\n        for path in sorted(self.local.rglob(\'*\')):\n            relative = path.relative_to(self.local)\n            if any(part.startswith(\'.\') for part in relative.parts) or not path.is_file():\n                continue\n            if path.is_symlink():\n                raise ValueError(\'Artifact symlinks are not supported\')\n            stat = path.stat()\n            stamps[str(relative)] = (stat.st_size, stat.st_mtime_ns)\n            if self.seen.get(str(relative)) != stamps[str(relative)]:\n                changed.append(path)\n        if not changed:\n            return\n        if len(checkpoints(self.remote)) != self.sequence:\n            raise RuntimeError(\'Another writer changed this backup directory. Stop the other notebook.\')\n        number = self.sequence + 1\n        stem = f\'{number:06d}-{uuid.uuid4().hex}\'\n        with tempfile.TemporaryDirectory() as td:\n            archive = Path(td) / (stem + \'.zip\')\n            with zipfile.ZipFile(archive, \'w\', zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n                for path in changed:\n                    z.write(path, str(path.relative_to(self.local)))\n            checksum = sha(archive)\n            destination = self.remote / archive.name\n            shutil.copyfile(archive, destination)\n            if sha(destination) != checksum:\n                raise IOError(\'Drive checkpoint verification failed; local results are preserved.\')\n            marker = {\'sequence\': number, \'archive\': archive.name, \'sha256\': checksum}\n            pending = self.remote / (stem + \'.pending\')\n            pending.write_text(json.dumps(marker))\n            os.replace(pending, self.remote / (stem + \'.ready.json\'))\n        self.seen = stamps\n        self.sequence = number\n        print(f\'DRIVE_BACKUP_OK checkpoint={number}\', flush=True)\n\n\n\ndef install_raw_journal(engine, backend_class, local, run_id):\n    """Save generated outputs before the scorer; never label this journal as scored."""\n    from contextlib import closing\n\n    prepare = engine.prepare_request\n    generate = backend_class.generate_stream\n    pending = {}\n\n    def prepare_request(*args, **kwargs):\n        request, record = prepare(*args, **kwargs)\n        pending[request[\'sample_id\']] = record\n        return request, record\n\n    def generate_stream(self, requests):\n        with closing(generate(self, requests)) as stream:\n            for output in stream:\n                record = pending.pop(output[\'sample_id\'])\n                payload = record | output | {\'journal_status\': \'GENERATED_UNSCORED\'}\n                # These paths are outside run/samples; final scoring never consumes them implicitly.\n                target = Path(local) / \'raw_journal\' / run_id / (record[\'inference_id\'] + \'.json\')\n                target.parent.mkdir(parents=True, exist_ok=True)\n                encoded = json.dumps(payload, ensure_ascii=False, allow_nan=False, sort_keys=True)\n                temporary = None\n                try:\n                    with tempfile.NamedTemporaryFile(mode=\'w\', encoding=\'utf-8\',\n                            dir=target.parent, prefix=\'.pending-\', delete=False) as stream_file:\n                        temporary = Path(stream_file.name)\n                        stream_file.write(encoded + \'\\n\')\n                        stream_file.flush()\n                        os.fsync(stream_file.fileno())\n                    os.link(temporary, target)  # Local disk: publish atomically without overwrite.\n                finally:\n                    if temporary is not None:\n                        temporary.unlink(missing_ok=True)\n                yield output\n\n    engine.prepare_request = prepare_request\n    backend_class.generate_stream = generate_stream\n\n\ndef validate_budget_2048(cfg, hw, baseline, original_validate):\n    """Allow one named output-budget experiment, retaining every upstream guard."""\n    import copy\n    expected = copy.deepcopy(baseline)\n    expected[\'protocol_id\'] = \'mmmu-val-v8-2048\'\n    expected[\'generation\'][\'max_new_tokens\'] = 2048\n    if cfg != expected:\n        raise ValueError(\'2048 experiment may change only protocol_id and max_new_tokens\')\n    original_validate(baseline, hw)\n\n\ndef install_budget_2048_protocol():\n    """Register a strict adapter extension without editing the pinned checkout."""\n    import yaml\n    import mmdl.runtime.contracts as contracts\n    repo = Path(contracts.__file__).resolve().parents[3]\n    baseline = yaml.safe_load((repo/\'configs/eval/mmmu_val_v8.yaml\').read_text())\n    original_validate = contracts.validate_configs\n\n    def validate(cfg, hw):\n        if cfg.get(\'protocol_id\') == \'mmmu-val-v8-2048\':\n            return validate_budget_2048(cfg, hw, baseline, original_validate)\n        return original_validate(cfg, hw)\n\n    contracts.validate_configs = validate\n\n\ndef preflight(repo, model_path, data_root, output, protocol=None):\n    """Validate all pinned inputs on CPU, without constructing a GPU model."""\n    import torch\n    import yaml\n    from transformers import AutoConfig, AutoProcessor\n    from mmdl.evaluation.datasets.mmmu import load_validation, separate_sample\n    from mmdl.evaluation.prompt import build_messages\n    from mmdl.evaluation.backends.input_preparation import prepare_protocol_inputs\n    from mmdl.runtime.artifacts import digest, write_json\n    from mmdl.runtime.contracts import MODEL_REVISION\n\n    torch.set_num_threads(4)\n    repo, output = Path(repo), Path(output)\n    cfg = yaml.safe_load(Path(protocol or repo / \'configs/eval/mmmu_val_v8.yaml\').read_text())\n    datasets, coverage = load_validation(Path(data_root))\n    processor = AutoProcessor.from_pretrained(str(model_path), revision=MODEL_REVISION,\n                                              local_files_only=True, trust_remote_code=False)\n    model_config = AutoConfig.from_pretrained(str(model_path), revision=MODEL_REVISION,\n                                             local_files_only=True, trust_remote_code=False)\n    processor.image_processor.size = {\'shortest_edge\': cfg[\'image\'][\'min_pixels\'],\n                                      \'longest_edge\': cfg[\'image\'][\'max_pixels\']}\n    records = []\n    for subject in sorted(datasets):\n        for row in datasets[subject]:\n            sample, images, _gold = separate_sample(row, subject)\n            if len(images) > 5:\n                raise ValueError(f"{sample[\'id\']}: exceeds fixed vLLM image limit")\n            messages = build_messages(sample, images, repo)\n            item, processed = prepare_protocol_inputs(processor, model_config, messages, images, cfg[\'image\'])\n            if item[\'input_tokens\'] + cfg[\'generation\'][\'max_new_tokens\'] > cfg[\'execution\'][\'max_model_len\']:\n                raise ValueError(f"{sample[\'id\']}: input plus output budget exceeds context")\n            records.append({\'id\': sample[\'id\'], \'subject\': subject, \'images\': len(images),\n                            \'question_type\': sample[\'question_type\'],\n                            \'input_tokens\': item[\'input_tokens\'], \'input_sha256\': item[\'input_sha256\']})\n            del item, processed, messages, images\n        print(f\'INPUT_PREFLIGHT {len(records)}/900\', flush=True)\n    if len(records) != 900 or len({r[\'id\'] for r in records}) != 900:\n        raise ValueError(\'Input preflight did not cover exactly 900 unique samples\')\n    result = {\'status\': \'PASS\', \'count\': 900, \'coverage\': coverage, \'protocol_sha256\': digest(cfg),\n              \'adapter_sha256\': sha(__file__), \'model_revision\': MODEL_REVISION,\n              \'max_input_tokens\': max(r[\'input_tokens\'] for r in records),\n              \'max_images\': max(r[\'images\'] for r in records), \'records\': records}\n    write_json(output, result)\n    return result\n\n\ndef run():\n    import sys\n    import mmdl.evaluation.engine as engine\n    from mmdl.evaluation.cli import main\n    from mmdl.runtime.artifacts import write_json\n\n    install_budget_2048_protocol()\n    local = Path(os.environ[\'MMDL_ARTIFACT_ROOT\'])\n    remote = Path(os.environ[\'MMDL_COLAB_BACKUP\'])\n    store = CheckpointStore(local, remote)\n    original_identity = engine.environment_identity\n    wrapper_hash = sha(__file__)\n    session_id = uuid.uuid4().hex\n\n    def identity(environment):\n        write_json(local / \'colab_sessions\' / f\'{session_id}.json\', {\n            \'environment\': environment, \'adapter_sha256\': wrapper_hash,\n            \'argv\': sys.argv, \'resume_policy\': \'ignore-host-kernel-release-only-v1\',\n        })\n        result = original_identity(environment)\n        result[\'platform\'] = {\'system\': platform.system(), \'machine\': platform.machine()}\n        result[\'colab_adapter_sha256\'] = wrapper_hash\n        # Check compatibility before RunWriter, to give a useful error without bypassing it.\n        if \'--resume\' in sys.argv:\n            from mmdl.runtime.artifacts import digest\n            run_id = sys.argv[sys.argv.index(\'--run-id\') + 1]\n            saved = json.loads((local / \'runs\' / run_id / \'run_manifest.json\').read_text())\n            if saved[\'identity\'][\'environment_sha256\'] != digest(result):\n                raise RuntimeError(\'Resume environment differs (GPU/driver/Python/packages/adapter). \'\n                                   \'Use the same environment; do not merge different GPUs or delete checks.\')\n        return result\n\n    from mmdl.evaluation.backends.vllm_backend import VLLMBackend\n    run_id = sys.argv[sys.argv.index(\'--run-id\') + 1]\n    install_raw_journal(engine, VLLMBackend, local, run_id)\n\n    original_writer = engine.RunWriter\n\n    class DriveWriter(original_writer):\n        def save(self, record):\n            super().save(record)\n            store.save()\n\n    engine.environment_identity = identity\n    engine.RunWriter = DriveWriter\n    support = local / \'colab_support\'\n    support.mkdir(parents=True, exist_ok=True)\n    shutil.copyfile(__file__, support / \'colab_runner.py\')\n    try:\n        main()\n    finally:\n        store.save()\n\n\nif __name__ == \'__main__\':\n    import sys\n    if sys.argv[1:2] == [\'preflight\']:\n        import argparse\n        parser = argparse.ArgumentParser()\n        for name in (\'repo\', \'model-path\', \'data-root\', \'output\'):\n            parser.add_argument(\'--\' + name, required=True, type=Path)\n        parser.add_argument(\'--protocol\', type=Path)\n        args = parser.parse_args(sys.argv[2:])\n        preflight(args.repo, args.model_path, args.data_root, args.output, args.protocol)\n    else:\n        run()\n'
RUNNER = Path('/content/colab_runner.py')
RUNNER.write_text(RUNNER_SOURCE)
import importlib.util
spec = importlib.util.spec_from_file_location('colab_backup', RUNNER)
backup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(backup)
if not LOCAL.exists():
    backup.restore(REMOTE, LOCAL)
else:
    print('현재 VM의 기존 로컬 결과를 유지합니다.')
print('복원된 본 평가 문항:', len(list((LOCAL/'runs'/f'{JOB_ID}-evaluation'/'samples').glob('*.json'))))


## 5. 모델·데이터 준비 및 GPU 설정
가중치와 데이터는 VM 로컬 디스크에 내려받습니다. 새 세션에서는 다시 받을 수 있지만 결과 백업은 유지됩니다. GPU 런타임에서 준비하는 시간에도 CU가 소모될 수 있습니다.

In [ ]:
call([PYTHON, '-m', 'mmdl.data.download', '--data-root', os.environ['MMDL_DATA_ROOT'],
      '--cache', os.environ['HF_HOME'], '--artifact-root', LOCAL, '--skip-pro'], cwd=REPO)
HW = Path('/content/colab-hardware.json')
hardware = dict(name='colab_gpu', placement='gpu_only', gpu_index=0,
    expected_vram_gib=GPU_MIB/1024, gpu_weight_cap_gib=GPU_MIB/1024-4,
    gpu_reserve_gib=3.0, cpu_weight_cap_gib=12.0, cpu_available_fraction=0.65,
    min_free_gpu_gib=12.0, allow_disk_offload=False, num_workers=0)
HW.write_text(json.dumps(hardware))
(LOCAL/'colab_support').mkdir(exist_ok=True)
shutil.copyfile(HW, LOCAL/'colab_support'/'hardware.json')
backup.CheckpointStore(LOCAL, REMOTE).save()
# Keep the pinned upstream checkout clean; preserve the experimental config separately.
PROTOCOL = LOCAL/'colab_support/mmmu_val_v8_2048.json'
protocol_source = subprocess.check_output([PYTHON, '-c',
    "import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))",
    REPO/'configs/eval/mmmu_val_v8.yaml'], text=True)
baseline_protocol = json.loads(protocol_source)
experiment_protocol = json.loads(protocol_source)
experiment_protocol['protocol_id'] = 'mmmu-val-v8-2048'
experiment_protocol['generation']['max_new_tokens'] = 2048
if PROTOCOL.exists():
    assert json.loads(PROTOCOL.read_text()) == experiment_protocol, '기존 실험 설정 불일치'
else:
    PROTOCOL.write_text(json.dumps(experiment_protocol, ensure_ascii=False, indent=2))
call([PYTHON, '-c',
    "import sys; sys.path.insert(0, '/content'); import colab_runner; "
    "colab_runner.install_budget_2048_protocol(); "
    "from mmdl.runtime.contracts import load_configs; "
    "cfg,hw=load_configs(sys.argv[1],sys.argv[2]); "
    "print('EXPERIMENT_CONFIG_VERIFIED', cfg['protocol_id'], cfg['generation'], cfg['execution'])",
    PROTOCOL, HW], cwd=REPO)
backup.CheckpointStore(LOCAL, REMOTE).save()
COMMON = [PYTHON, RUNNER, '--protocol', PROTOCOL,
    '--hardware', HW, '--model-ref', REPO/'manifests/models/baseline.json',
    '--model-path', 'Qwen/Qwen3-VL-4B-Instruct',
    '--data-root', Path(os.environ['MMDL_DATA_ROOT'])/'evaluation/mmmu',
    '--artifact-root', LOCAL, '--public-root', LOCAL/'public',
    '--require-commit', COMMIT, '--no-download']
def evaluate(run_id, extra):
    args = COMMON + ['--run-id', run_id] + extra
    if (LOCAL/'runs'/run_id).exists():
        args += ['--resume']
    call(args, cwd=REPO)


## 6. 3문항 사전 실행
메모리·추론·Drive 백업이 되는지 확인합니다. 본 평가와 분리된 결과이며 900문항 성적에는 합치지 않습니다. 일부 문항도 긴 출력 때문에 오래 걸릴 수 있습니다.

In [ ]:
RUN_SMOKE = False  # 기존 3문항 GPU 검증을 반복하려면 True
if RUN_SMOKE:
    evaluate(JOB_ID+'-smoke', ['--mode', 'smoke', '--run-role', 'smoke', '--limit', '3'])
else:
    print('별도 smoke 생략. 전체 실행의 초반 결과를 관찰하세요.')


## 7. 비용 계산 및 900문항 평가
**아래 실행 셀은 기본적으로 꺼져 있습니다.** 먼저 Colab 리소스 화면에서 시간당 CU를 확인하세요. 3문항은 기능 확인용으로, 전체 시간 예측에는 대표성이 부족합니다. 과목과 출력 길이에 따라 소요가 크게 달라집니다.

900문항 실행 중 세션이 종료되면 같은 JOB_ID로 1–5단계 후 7단계를 다시 실행하세요. 완료 문항을 건너뜁니다. 같은 GPU/드라이버/패키지가 아니면 재개하지 않습니다. 작업을 멈출 때 백업 완료 메시지를 확인하세요. 단순히 브라우저 탭을 닫는 것은 런타임 종료와 다릅니다.

In [ ]:
# 실제 잔액과 요율은 Colab 리소스 화면에서 관측하여 로컬 실행 상태에 기록합니다.
# 확인하지 않은 가격·시간을 추정 실측값으로 표시하지 않습니다.
print('출력 한도 2,048 / 전체 컨텍스트 40,960 / 900문항 / L4 / V8')
print('JOB_ID:', JOB_ID, '| Drive:', REMOTE)


In [ ]:
RUN_FULL = False  # 전체 실행 시작 시 True
AUTO_DISCONNECT = True  # 성공/오류 모두 최종 백업이 확인된 경우 종료
if RUN_FULL:
    import uuid
    from datetime import datetime, timezone
    RUN_DIR = LOCAL/'runs'/f'{JOB_ID}-evaluation'
    state = {'job_id': JOB_ID, 'status': 'RUNNING', 'started_at': datetime.now(timezone.utc).isoformat(), 'max_new_tokens': 2048, 'max_model_len': 40960}
    try:
        # CPU preprocessing only; no GPU model is constructed until this passes.
        model_snapshot = Path(os.environ['HF_HOME'])/'hub/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17'
        call([PYTHON, RUNNER, 'preflight', '--repo', REPO, '--model-path', model_snapshot,
              '--data-root', Path(os.environ['MMDL_DATA_ROOT'])/'evaluation/mmmu',
              '--protocol', PROTOCOL, '--output', LOCAL/'colab_support/input_preflight.json'], cwd=REPO)
        backup.CheckpointStore(LOCAL, REMOTE).save()
        evaluate(JOB_ID+'-evaluation', ['--mode', 'full', '--run-role', 'evaluation', '--job-id', JOB_ID])
        summary = json.loads((RUN_DIR/'summary.json').read_text())
        assert summary['status'] == 'COMPLETE', '미완료 실행: 원문을 보존하고 원인 확인'
        call([PYTHON, '-m', 'scripts.validate_run', '--run-dir', RUN_DIR,
              '--public-dir', LOCAL/'public'/RUN_DIR.name], cwd=REPO)
        # A failed prior rescore directory remains intact; every attempt gets its own path.
        SCORED = LOCAL/'rescores'/f'{JOB_ID}-v8-{uuid.uuid4().hex[:12]}'
        call(['bash', 'scripts/rescore.sh', '--artifact-root', LOCAL,
              '--run-id', RUN_DIR.name, '--parser', 'team-final-answer-v8',
              '--output-dir', SCORED], cwd=REPO, env=dict(os.environ, MMDL_PYTHON=str(PYTHON)))
        call([PYTHON, '-m', 'scripts.validate_rescore', '--source-run', RUN_DIR,
              '--scored-dir', SCORED], cwd=REPO)
        state.update(status='COMPLETE', scored_dir=str(SCORED.relative_to(LOCAL)))
        print('900문항 원문·V8 재채점 검증 완료')
        print(json.dumps(summary, ensure_ascii=False, indent=2))
    except BaseException as exc:
        state.update(status='FAILED', error_type=type(exc).__name__, error=str(exc))
        raise
    finally:
        state['ended_at'] = datetime.now(timezone.utc).isoformat()
        # Failure does not bypass cleanup. Never destroy local data after a failed backup.
        (LOCAL/'colab_support/lifecycle.json').write_text(json.dumps(state, ensure_ascii=False, indent=2))
        try:
            backup.CheckpointStore(LOCAL, REMOTE).save()
        except BaseException:
            print('최종 백업 실패: 런타임을 유지합니다. 로컬 결과를 보존하고 백업부터 복구하세요.')
            raise
        else:
            print('최종 상태:', state['status'], '| Drive 백업 확인:', REMOTE)
            if AUTO_DISCONNECT:
                from google.colab import runtime
                runtime.unassign()
else:
    print('본 평가 대기: RUN_FULL=True로 바꾸면 입력 검사 → 900문항 → 재채점 → 백업·종료를 실행합니다.')


## 8. 보존된 결과 확인

최종 실행 셀은 900문항 완료 후 원본 검증기와 독립 V8 재채점 검증기를 실행합니다. 세션별 시간은 각 기록에 남고, 원본 summary의 `*_this_invocation`은 마지막 호출 시간입니다. 여러 세션의 전체 소요 시간으로 잘못 보고하지 마세요. 기존 RTX3090 보고서의 62%는 이번 결과가 아닙니다.


결과는 Drive의 동일 JOB_ID 백업에 있습니다. `colab_support/lifecycle.json`에서 COMPLETE/FAILED와 재채점 결과 위치를 확인합니다.
FAILED를 완료 성적으로 사용하지 마세요. 채점 전 원문은 `raw_journal/<run-id>/`에도 보존됩니다. 파서 예외가 나면 실행은 멈추며, 이 원문은 자동으로 기존 재채점 명령에 합쳐지지 않습니다.
